In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

In [2]:
data = pd.read_stata('speech_data.dta')

## Descriptive Statistics

In [ ]:
variables = ['vote_against', 'gal_tan','v4_d', 'nordic_d', 'pigs_d', 'GDP_per_capita', 
             'gender_num', 'newcomer_num', 'Tenure']

stats_df = pd.DataFrame(index=variables, 
                        columns=['Observations', 'Mean', 'SD', 'Min', 'Max'])

for var in variables:
    stats_df.loc[var, 'Observations'] = data[var].count()
    stats_df.loc[var, 'Mean'] = data[var].mean()
    stats_df.loc[var, 'SD'] = data[var].std()
    stats_df.loc[var, 'Min'] = data[var].min()
    stats_df.loc[var, 'Max'] = data[var].max()

stats_df = stats_df.apply(pd.to_numeric)
stats_df = stats_df.round(2)

stats_df

## Evaluation of LLM Classification Performance

### First-Stage Classification Evaluation

In [ ]:
# Load dataset with human-annotated speeches
evaluation_df = pd.read_csv('evaluation_df.csv')

# Add predicted classes
evaluation_df = evaluation_df.merge(
    data[['ID', 'framing_final']],
    on='ID',
    how='left'
)

# Rename classes
evaluation_df = evaluation_df.replace({
    "denialist": "Denialist",
    "fatalistic": "Fatalistic",
    "none": "None",
    "solution-oriented": "Solution-oriented"
})

def calculate_framing_metrics(df):
    y_true = df['framing_manual']
    y_pred = df['framing_final']

    accuracy = accuracy_score(y_true, y_pred)

    labels = sorted(list(set(y_true) | set(y_pred)))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)

    report = classification_report(y_true, y_pred, labels=labels, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    print("\nClassification Report:")
    print(report_df.round(4))

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        cm_df,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar_kws={'label': 'Count'},
        annot_kws={"size": 14},
        ax=ax
    )

    ax.set_xlabel('Predicted Framings', fontsize=17)
    ax.set_ylabel('Human-Labeled Framings', fontsize=17)
    ax.tick_params(axis='both', labelsize=13)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=13)
    cbar.ax.set_ylabel('', fontsize=14)

    plt.tight_layout()

    return accuracy, cm_df, report_df


result = calculate_framing_metrics(evaluation_df)

### Second-Stage Classification Evaluation

In [ ]:
# Load dataset with human-annotated speeches
evaluation_df_detailed = pd.read_csv('evaluation_df_detailed.csv')

# Add predicted classes
evaluation_df_detailed = evaluation_df_detailed.merge(
    data[['ID', 'framing_final_detailed']],
    on='ID',
    how='left'
)

evaluation_df_detailed = evaluation_df_detailed.replace({
    "Solution-oriented_passive": "Solution-oriented Passive",
    "Solution-oriented_proactive": "Solution-oriented Proactive"
})

def calculate_framing_metrics(df):
    y_true = df['framing_manual']
    y_pred = df['framing_final_detailed']

    accuracy = accuracy_score(y_true, y_pred)

    labels = sorted(list(set(y_true) | set(y_pred)))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)

    report = classification_report(y_true, y_pred, labels=labels, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    print("\nClassification Report:")
    print(report_df.round(4))

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm_df,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar_kws={'label': 'Count'},
        annot_kws={"size": 14},
        ax=ax
    )
    ax.set_xlabel('Predicted Framings', fontsize=17)
    ax.set_ylabel('Human-Labeled Framings', fontsize=17)
    ax.tick_params(axis='both', labelsize=13)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=13)
    cbar.ax.set_ylabel('', fontsize=14)
    plt.tight_layout()
    
    return accuracy, cm_df, report_df

result = calculate_framing_metrics(evaluation_df_detailed)